[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cstecker/politicsRLab/blob/main/In%20welcher%20Blase%20bin%20ich%20denn%20hier%20gelandet%20-%202026-10.ipynb)

# In welcher Blase bin ich denn hier gelandet?

**Kursumfrage BRD · Wintersemester 2026/27**

Wer sitzt mit uns im Hörsaal? Welche politischen Positionen sind häufig, welche selten? Wir untersuchen unsere Kursumfrage und lernen dabei, Häufigkeiten zu zählen, Mittelwert und Modalwert zu bestimmen und eine einfache Hypothese zu prüfen.

Führen Sie die Zellen der Reihe nach mit dem Play-Button oder **Strg + Enter** aus. Wenn Sie eine Zelle ändern, führen Sie auch die davon abhängigen Zellen erneut aus.

**Bevor wir loslegen:** Vermuten Sie, dass sich die Antworten eher links, in der Mitte oder rechts häufen? Begründen Sie Ihre Erwartung in einem Satz.

## 1. R vorbereiten

Das Paket **tidyverse** enthält Werkzeuge zum Einlesen, Zählen und Zeichnen. Die erste Zelle installiert es bei Bedarf und lädt es mit `library()`.

In [ ]:
if (!requireNamespace("tidyverse", quietly = TRUE)) {
  install.packages("tidyverse", repos = "https://cloud.r-project.org")
}
library(tidyverse)
options(repr.plot.width = 10, repr.plot.height = 5)

## 2. Daten laden und kennenlernen

Die Übungsdatei enthält abgeschlossene Interviews mit Einwilligung. Persönliche Codes, IDs, Zeitstempel, Geburtsorte und Freitexte sind entfernt; seltene Kategorien geschützt. Jede Zeile ist ein Interview, jede Spalte eine Variable.

Die nächste Zelle lädt die Datei von GitHub. Eine bereits in der Laufzeit vorhandene Übungsdatei wird weiterverwendet.

In [ ]:
datei <- "survey2026-10_colab.rds"
if (!file.exists(datei)) {
  download.file(
    "https://raw.githubusercontent.com/cstecker/politicsRLab/main/data/survey2026-10_colab.rds",
    datei, mode = "wb"
  )
}

`readRDS()` liest die gespeicherte R-Tabelle. Mit `<-` geben wir ihr den Namen `survey`.

In [ ]:
survey <- readRDS(datei)
stand <- attr(survey, "kursumfrage")$snapshot_date

`cat()` gibt einen kurzen Text aus. `nrow()` zählt die Zeilen, hier also die Interviews.

In [ ]:
cat("Datenstand:", stand, "· Interviews:", nrow(survey), "\n")

`glimpse()` zeigt Variablennamen, Datentypen und erste Werte.

In [ ]:
glimpse(survey)

`head()` zeigt die ersten sechs Zeilen als Tabelle.

In [ ]:
head(survey)

| Variable | Bedeutung / Skalenenden |
|---|---|
| `geschlecht` | Weiblich / Männlich; weitere oder seltene Angaben: NA |
| `vote` | Wahlabsicht; seltene Angaben zusammengefasst |
| `lire_self` | 1 = links, 11 = rechts |
| `econ_self` | 1 = mehr Sozialstaat, 11 = weniger Sozialstaat |
| `immi_self` | 1 = Zuzug erleichtern, 11 = Zuzug einschränken |
| `klim_self` | 1 = Vorrang Klimaschutz, 11 = Vorrang Wirtschaftswachstum |
| `euin_self` | 1 = mehr EU-Kompetenzen, 11 = weniger EU-Kompetenzen |

`NA` bezeichnet einen fehlenden Wert. Bei den Positionsskalen wurde die Frage übersprungen; bei Geschlecht und Wahlabsicht kann zusätzlich eine Angabe unterdrückt worden sein. **NA ist weder Null noch die politische Mitte.**

`count()` zählt, wie häufig die einzelnen Antworten vorkommen. Die Pipe `|>` gibt das Ergebnis links an den nächsten Befehl weiter.

In [ ]:
survey |> count(geschlecht)

## 3. Welche Parteien würden die Teilnehmenden wählen?

`sort = TRUE` ordnet die Häufigkeiten absteigend. Die Tabelle enthält auch fehlende Werte.

In [ ]:
survey |> count(vote, sort = TRUE)

Nun berechnen wir Prozentanteile. `filter()` wählt Zeilen aus, `mutate()` ergänzt Spalten. Für den Nenner verwenden wir nur veröffentlichte Wahlabsichten; „Nichtwahl“ bleibt als inhaltliche Antwort enthalten.

In [ ]:
vote_share <- survey |>
  filter(!is.na(vote)) |>              # Fehlende Wahlabsichten ausschließen.
  count(vote, sort = TRUE) |>          # Antworten zählen und absteigend sortieren.
  mutate(
    share = n / sum(n),                # Häufigkeit durch alle gültigen Antworten teilen.
    Prozent = round(100 * share, 1)    # In Prozent umrechnen und auf eine Nachkommastelle runden.
  )

Ein Objektname allein zeigt die gespeicherte Tabelle an.

In [ ]:
vote_share

`sum()` addiert Werte. `is.na()` erkennt fehlende Werte; deren Summe zählt hier die fehlenden Angaben.

In [ ]:
cat("Veröffentlichte Wahlabsichten:", sum(vote_share$n),
    "· fehlend/unterdrückt:", sum(is.na(survey$vote)), "\n")

`ggplot()` beginnt eine Grafik, `aes()` legt die Achsen fest. `geom_col()` zeichnet die berechneten Balken, `reorder()` sortiert ihre Reihenfolge. `coord_flip()` dreht die Grafik, damit die Parteinamen gut lesbar sind.

In [ ]:
p_vote <- ggplot(vote_share, aes(x = reorder(vote, share), y = share)) +
  geom_col(fill = "#9C1C26") +
  coord_flip() +
  scale_y_continuous(labels = scales::label_percent()) +
  labs(title = "Wahlabsicht in der Kursumfrage",
       x = NULL, y = "Anteil der veröffentlichten Wahlabsichten",
       caption = paste("Basis:", sum(vote_share$n), "Antworten; einschließlich Nichtwahl.")) +
  theme_minimal(base_size = 13)
p_vote

## 4. Wo liegen wir auf der Links-Rechts-Skala?

Die Skala reicht von **1 = links** bis **11 = rechts**. `filter()` behält für diesen Abschnitt nur gültige Links-Rechts-Antworten. Der übrige Datensatz bleibt erhalten.

In [ ]:
lire_daten <- survey |>
  filter(!is.na(lire_self))

`summarise()` fasst viele Zeilen zu einer Ergebniszeile zusammen. Der **Mittelwert** addiert alle Antworten und teilt durch ihre Anzahl. Dabei behandeln wir die Abstände zwischen den Skalenstufen als gleich groß.

In [ ]:
lire_mittelwert <- lire_daten |>
  summarise(
    Anzahl = n(),                     # Zahl der gültigen Antworten.
    Mittelwert = mean(lire_self)       # Rechnerischer Durchschnitt.
  )
lire_mittelwert

Der **Modalwert** ist die häufigste Kategorie. Wir zählen die Antworten und behalten alle Kategorien mit der höchsten Häufigkeit. Bei Gleichstand gibt es mehrere Modalwerte.

In [ ]:
modalwerte <- lire_daten |>
  count(lire_self) |>
  filter(n == max(n))                  # Alle Kategorien mit maximaler Häufigkeit behalten.
modalwerte

### Absolute Häufigkeiten als Balkendiagramm

`select()` wählt die untersuchte Variable und nennt sie hier `Antwort`. Damit müssen Sie für die Quizaufgabe nur **eine Variablenangabe** ändern. `complete()` ergänzt nicht gewählte Kategorien mit der Häufigkeit Null.

In [ ]:
haeufigkeiten <- survey |>
  select(Antwort = lire_self) |>       # Hier wird die untersuchte Variable ausgewählt.
  filter(!is.na(Antwort)) |>           # Fehlende Antworten ausschließen.
  count(Antwort) |>                    # Absolute Häufigkeiten zählen.
  complete(Antwort = 1:11, fill = list(n = 0))  # Alle elf Kategorien anzeigen.

`geom_text()` schreibt die absoluten Häufigkeiten über die Balken. `breaks = 1:11` zeigt jede Antwortkategorie auf der x-Achse.

In [ ]:
p_haeufigkeiten <- ggplot(haeufigkeiten, aes(x = Antwort, y = n)) +
  geom_col(fill = "#657A86") +
  geom_text(aes(label = n), vjust = -0.4) +  # Die Anzahl über jedem Balken anzeigen.
  scale_x_continuous(breaks = 1:11) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.15))) +
  labs(title = "Selbsteinstufung: absolute Häufigkeiten",
       x = "Antwortkategorie (1–11)", y = "Anzahl Antworten",
       caption = paste("Datenstand:", stand, "· gültige Antworten:", sum(haeufigkeiten$n))) +
  theme_minimal(base_size = 13)
p_haeufigkeiten

### Quiz: Ändern Sie die Variable

Ersetzen Sie in der Datenzelle oben **nur `lire_self` durch `immi_self`**. Führen Sie anschließend diese Datenzelle und die Grafikzelle erneut aus.

Die Zuzugsskala reicht von **1 = Zuzug erleichtern** bis **11 = Zuzug einschränken**.

- Wie viele Teilnehmende wählten Kategorie 8?
- Welche Kategorie ist am häufigsten?
- Welcher Balken ist höher: Kategorie 1 oder Kategorie 11?

Lesen Sie die absoluten Zahlen an den Balken ab und gleichen Sie sie mit der Tabelle ab.

In [ ]:
haeufigkeiten

## 5. Hypothese: Sind männliche Studierende rechter als weibliche?

Wir untersuchen dieselbe Erwartung mit zwei unterschiedlichen Messungen: **Wahlabsicht** und **Links-Rechts-Selbsteinstufung**. Halten Sie Ihre Erwartung fest, bevor Sie die Ergebnisse ansehen.

Ein gerichteter Test prüft hier einen höheren Anteil beziehungsweise Mittelwert bei Männern. Diese Richtung wird **vor** dem Test festgelegt. Der p-Wert beschreibt, wie ungewöhnlich das Ergebnis oder ein stärkeres Ergebnis unter der Nullhypothese wäre. Er ist nicht die Wahrscheinlichkeit, dass unsere Hypothese stimmt.

Als einfache Entscheidungsregel verwenden wir `p < 0,05`: Dann sprechen wir von statistischer Evidenz für die gerichtete Alternative unter den Testannahmen. Ein größerer p-Wert beweist keine Gleichheit. Die beiden Tests sind Übungen am Kursdatensatz; die Teilnehmenden sind keine Zufallsstichprobe der deutschen Studierenden, und ein Unterschied belegt keine Ursache.

### 5a. Wahlabsicht: CDU/FDP häufiger bei Männern?

Wir fassen **CDU/FDP** und **SPD/Grüne/Linke** zu zwei Gruppen zusammen. Das ist eine grobe Operationalisierung von „rechter wählen“, keine eindeutige Links-Rechts-Messung jeder Partei.

Nichtwahl, fehlende Angaben und die politisch nicht eindeutig zuordenbare Sammelkategorie „Andere / selten genannt“ werden für diesen Test ausgeschlossen. Die AfD ist in den aktuellen Übungsdaten nicht separat ausgewiesen. Der Test trifft deshalb eine Aussage über **CDU/FDP gegenüber SPD/Grünen/Linken**, nicht über sämtliche rechten Wahlabsichten.

**Nullhypothese:** Männer wählen nicht häufiger CDU/FDP als Frauen. **Alternative:** Der CDU/FDP-Anteil ist bei Männern höher.

In [ ]:
wahldaten <- survey |>
  filter(
    geschlecht %in% c("Männlich", "Weiblich"),
    vote %in% c("CDU", "FDP", "SPD", "Grüne", "Die Linke")
  ) |>
  mutate(
    wahlgruppe = if_else(vote %in% c("CDU", "FDP"), "CDU/FDP", "SPD/Grüne/Linke"),
    # Für den gerichteten Test stehen Männer und CDU/FDP jeweils an erster Stelle.
    geschlecht = factor(geschlecht, levels = c("Männlich", "Weiblich")),
    wahlgruppe = factor(wahlgruppe, levels = c("CDU/FDP", "SPD/Grüne/Linke"))
  )

`if_else()` ordnet eine CDU/FDP-Antwort der ersten und die übrigen berücksichtigten Parteien der zweiten Wahlgruppe zu. `factor()` legt die Reihenfolge der Kategorien fest; das ist für die Testrichtung wichtig.

Zuerst beschreiben wir die Daten. `group_by()` berechnet die Prozentanteile getrennt nach Geschlecht; `ungroup()` beendet diese Gruppierung.

In [ ]:
wahlanteile <- wahldaten |>
  count(geschlecht, wahlgruppe) |>
  group_by(geschlecht) |>
  mutate(Prozent = round(100 * n / sum(n), 1)) |>
  ungroup()
wahlanteile

`nrow()` zeigt, wie viele Interviews nach den Filtern übrig bleiben.

In [ ]:
cat("Für diesen Test verwendet:", nrow(wahldaten),
    "· ausgeschlossen:", nrow(survey) - nrow(wahldaten), "\n")

`xtabs()` erstellt die Kreuztabelle mit Geschlecht in den Zeilen und Wahlgruppe in den Spalten.

In [ ]:
wahltabelle <- xtabs(~ geschlecht + wahlgruppe, data = wahldaten)
wahltabelle

`fisher.test()` prüft den Unterschied in der Kreuztabelle auch bei kleinen Häufigkeiten. `alternative = "greater"` legt die erwartete Richtung fest: Die erste Geschlechtsgruppe (Männer) soll häufiger die erste Wahlgruppe (CDU/FDP) wählen. Lesen Sie zunächst die Ergebniszeile **p-value**.

In [ ]:
wahl_test <- fisher.test(wahltabelle, alternative = "greater")
wahl_test

### 5b. Links-Rechts-Skala: höherer Mittelwert bei Männern?

Hier verwenden wir direkt die Selbsteinstufung: Höhere Werte bedeuten „weiter rechts“. Fehlende und unterdrückte Geschlechtsangaben sowie fehlende Skalenantworten werden für diesen Vergleich ausgeschlossen.

**Nullhypothese:** Der Mittelwert bei Männern ist nicht höher als bei Frauen. **Alternative:** Der Mittelwert bei Männern ist höher.

In [ ]:
lr_daten <- survey |>
  filter(geschlecht %in% c("Männlich", "Weiblich"), !is.na(lire_self)) |>
  mutate(geschlecht = factor(geschlecht, levels = c("Männlich", "Weiblich")))

Wir vergleichen zunächst die Fallzahlen und Mittelwerte. `.groups = "drop"` hebt die Gruppierung nach der Zusammenfassung auf.

In [ ]:
lr_mittelwerte <- lr_daten |>
  group_by(geschlecht) |>
  summarise(Anzahl = n(), Mittelwert = mean(lire_self), .groups = "drop")
lr_mittelwerte

`t.test()` vergleicht die Mittelwerte. Die Formel `lire_self ~ geschlecht` heißt: „Vergleiche die Links-Rechts-Werte nach Geschlecht.“ Männer stehen an erster Stelle, deshalb prüft `greater` einen höheren Mittelwert bei Männern. Der standardmäßige **Welch-t-Test** setzt keine gleichen Gruppenvarianzen voraus; wie beim Mittelwert behandeln wir die Skalenabstände als gleich groß.

In [ ]:
lr_test <- t.test(lire_self ~ geschlecht, data = lr_daten, alternative = "greater")
lr_test

### Beide Ergebnisse lesen

Mit `$p.value` lesen wir den p-Wert aus einem Testergebnis. Vergleichen Sie beide p-Werte mit 0,05 und nennen Sie dazu jeweils den beobachteten Gruppenunterschied. Die Tests verwenden wegen der unterschiedlichen Filter nicht genau dieselben Interviews.

In [ ]:
tibble(
  Test = c("Wahlabsicht: CDU/FDP", "Links-Rechts-Selbsteinstufung"),
  p_Wert = c(wahl_test$p.value, lr_test$p.value)
)

## 6. Politik hat mehr als eine Achse

Hängen Einstellungen zum Sozialstaat und zum Zuzug zusammen? `count(econ_self, immi_self)` zählt Kombinationen beider Antworten.

In [ ]:
kombinationen <- survey |>
  filter(!is.na(econ_self), !is.na(immi_self)) |>
  count(econ_self, immi_self)

Jeder Punkt steht für eine Antwortkombination. `size = n` lässt häufigere Kombinationen größer erscheinen; `scale_size_area()` skaliert die Punktfläche mit der Anzahl.

In [ ]:
p_dimensionen <- ggplot(kombinationen, aes(x = econ_self, y = immi_self, size = n)) +
  geom_point(color = "#9C1C26", alpha = 0.65) +
  scale_size_area(max_size = 13, breaks = c(1, 5, 10)) +
  scale_x_continuous(breaks = 1:11, limits = c(0.5, 11.5)) +
  scale_y_continuous(breaks = 1:11, limits = c(0.5, 11.5)) +
  labs(title = "Sozialstaat und Zuzug: zwei politische Dimensionen",
       x = "1 = mehr Sozialstaat, 11 = weniger Sozialstaat",
       y = "1 = Zuzug erleichtern, 11 = Zuzug einschränken",
       size = "Anzahl",
       caption = paste("Basis:", sum(kombinationen$n), "Interviews mit beiden Antworten.")) +
  theme_minimal(base_size = 13)
options(repr.plot.height = 6)
p_dimensionen

## 7. Jetzt Sie

Wählen Sie eine andere Positionsskala und stellen Sie eine kleine Frage. Beginnen Sie mit einer Erwartung, zählen Sie die Antworten und prüfen Sie Ihre Erwartung an der Tabelle oder Grafik.

**Zum Abschluss ohne zurückzublättern:** Was unterscheidet Mittelwert und Modalwert? Was bedeutet ein kleiner p-Wert, und was lässt sich daraus nicht schließen?

In [ ]:
survey |> count(klim_self)

# Hier ist Platz für Ihre eigene Auswertung.